<h1>RAZ Systems</h1>

## Problem Statement

You're building a **smart banking assistant** as a LangGraph agent. Unlike the single-tool agents earlier in this curriculum, this one needs **three distinct capabilities**, and it has to decide for itself which one a given question needs:

1. **Account lookup** — exact-match SQLite query, by account number (same data as the AutoGen banking notebook)
2. **Policy retrieval (RAG)** — embedding-similarity search over a small set of bank policy documents (overdraft, closure, fraud, etc.)
3. **Live web search** — via the Serper API, for questions the bank's own data can't answer (e.g. "what's the current Fed interest rate?")

The point of this assignment isn't learning three new things — you've already built each of these individually. The point is proving you understand **how LangGraph's `ToolNode` doesn't care how many tools it wraps, or how different they are internally** — a SQLite query, a RAG retrieval, and an HTTP call to Serper are all just "a Python function with a docstring" from `ToolNode`'s point of view.

By the end, one LangGraph agent will look at a question, decide which of the three tools (or none) it needs, call it, and answer — entirely through `bind_tools` + `ToolNode` + `tools_condition`, the same pattern from `v3_toolnode.py`.

This notebook follows the same shape as the rest of this curriculum: a few **concepts** built up step by step (all given to you, working), then an **Assignment** section with `TODO`s, then the full **Solution** at the bottom.

### Setup

In [ ]:
####!pip install -q langgraph langchain-openai langchain-core requests python-dotenv

In [ ]:
import os
import re
import math
import sqlite3

import requests
from dotenv import load_dotenv
from typing import Annotated, TypedDict

from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv(override=True)

SERPER_API_KEY = os.getenv("SERPER_API_KEY")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

### First concept: the bank account database (given, unchanged from the AutoGen banking notebook)

Same SQLite setup as before — this is data, not the lesson, so it's fully provided.

In [ ]:
if os.path.exists("bank.db"):
    os.remove("bank.db")

conn = sqlite3.connect("bank.db")
c = conn.cursor()
c.execute("""
CREATE TABLE bank_accounts (
    account_number TEXT PRIMARY KEY,
    customer_name TEXT,
    balance REAL
)
""")
conn.commit()
conn.close()


def save_account_balance(account_number, customer_name, balance):
    conn = sqlite3.connect("bank.db")
    c = conn.cursor()
    c.execute("""
    REPLACE INTO bank_accounts
    (account_number, customer_name, balance)
    VALUES (?, ?, ?)
    """, (account_number, customer_name, balance))
    conn.commit()
    conn.close()


save_account_balance("ACC1001", "Ajaz Pasha", 15000.75)
save_account_balance("ACC1002", "Aiza Khan", 24500.00)
save_account_balance("ACC1003", "Numan Khan", 8750.50)
save_account_balance("ACC1004", "Mohammed", 32000.25)
save_account_balance("ACC1005", "N Jahan", 12600.00)

print("✅ bank.db ready with 5 accounts")

### Second concept: the bank policy knowledge base (given, unchanged from the RAG notebook)

Same 5 policy documents as the AutoGen + RAG assignment. Given fully worked since the documents themselves aren't the lesson here either.

In [ ]:
BANK_POLICIES = [
    {
        "id": "overdraft",
        "title": "Overdraft Policy",
        "content": (
            "Customers may overdraw their account by up to $500 without incurring a fee, "
            "provided the account has been open for at least 90 days and has no prior overdraft "
            "violations in the last 12 months. Overdrafts beyond $500 incur a flat $35 fee per "
            "occurrence. Accounts overdrawn for more than 30 consecutive days may be suspended."
        ),
    },
    {
        "id": "closure",
        "title": "Account Closure Policy",
        "content": (
            "Customers may close their account at any time by visiting a branch or submitting a "
            "written request. Accounts with a negative balance must be brought to zero or positive "
            "before closure. Closure requests are processed within 5 business days, and any remaining "
            "balance is issued via check to the address on file."
        ),
    },
    {
        "id": "fraud",
        "title": "Fraud Reporting Policy",
        "content": (
            "Suspected fraudulent activity must be reported within 60 days of the statement date on "
            "which the unauthorized transaction appears. Customers are not liable for confirmed "
            "fraudulent transactions reported within this window. The bank will issue a provisional "
            "credit within 10 business days while the investigation is ongoing."
        ),
    },
    {
        "id": "interest",
        "title": "Savings Interest Policy",
        "content": (
            "Savings accounts accrue interest daily at the published annual percentage yield (APY) "
            "and are credited monthly. The APY is variable and may change with 30 days' written "
            "notice. A minimum balance of $100 is required to earn interest in any given month."
        ),
    },
    {
        "id": "dormant",
        "title": "Dormant Account Policy",
        "content": (
            "An account with no customer-initiated transactions for 12 consecutive months is "
            "classified as dormant. Dormant accounts incur no fees but may require identity "
            "re-verification before further transactions are permitted. Accounts dormant for 5 "
            "years or more may be subject to state unclaimed-property reporting requirements."
        ),
    },
]

# Pre-compute embeddings once -- same idea as the HR Handbook RAG system,
# just held in memory instead of Supabase/pgvector.
_policy_texts = [p["content"] for p in BANK_POLICIES]
_policy_vectors = embeddings.embed_documents(_policy_texts)
for p, v in zip(BANK_POLICIES, _policy_vectors):
    p["embedding"] = v

print(f"✅ {len(BANK_POLICIES)} policy documents embedded")

## Assignment

Everything above is given to you, fully working. Your job is to build **three tools** and wire them into a LangGraph agent using `ToolNode` + `tools_condition` — the same pattern as `v3_toolnode.py`. Each TODO below explains exactly what's needed. The full solution is at the bottom if you get stuck.

### TODO 1 — `get_account_balance` tool

Wrap the SQLite lookup from the AutoGen banking notebook as a LangGraph `@tool`. This one's a direct port — same query, same return format — just decorated correctly so an LLM can call it.

In [ ]:
@tool
def get_account_balance(account_number: str) -> str:
    """Look up a customer's name and balance by their bank account number (e.g. ACC1003)."""
    # TODO 1: connect to bank.db, SELECT customer_name and balance WHERE
    # account_number matches, and return a string like:
    #   f"Customer: {customer_name}, Balance: ${balance}"
    # or "Account not found" if no row matches. Don't forget to close the connection.
    raise NotImplementedError("TODO 1: implement get_account_balance")

### TODO 2 — `search_bank_policies` tool (RAG)

Wrap the cosine-similarity retrieval from the RAG notebook as a second `@tool`. Given a query, embed it, compare against every policy's pre-computed embedding in `BANK_POLICIES`, and return the best match.

In [ ]:
def cosine_similarity(a: list, b: list) -> float:
    # TODO 2a: implement cosine similarity between two equal-length vectors.
    raise NotImplementedError("TODO 2a: implement cosine_similarity")


@tool
def search_bank_policies(query: str) -> str:
    """Search the bank's internal policy documents (overdraft, account closure, fraud reporting, interest, dormant accounts) for information relevant to the query."""
    # TODO 2b:
    #   1. embeddings.embed_query(query) to get the query's embedding
    #   2. score every policy in BANK_POLICIES with cosine_similarity against
    #      that policy's "embedding"
    #   3. find the highest-scoring policy
    #   4. return f"{best['title']}: {best['content']}"
    raise NotImplementedError("TODO 2b: implement search_bank_policies")

### TODO 3 — `search_web` tool (Serper)

Wrap a live Serper web search as a third `@tool` — same `requests.post("https://google.serper.dev/search", ...)` pattern used throughout this curriculum (`search_flights` / `search_hotels` in the travel-agent notebooks). This is the tool the agent should reach for when a question is about something the bank's own data can't possibly contain (current interest rates set by a central bank, recent financial news, etc.) — as opposed to the bank's *own* policies, which live in `search_bank_policies`.

In [ ]:
@tool
def search_web(query: str) -> str:
    """Search the live web for current information not contained in the bank's own records or policies -- e.g. current market interest rates, recent financial news, or general finance questions."""
    # TODO 3: POST to https://google.serper.dev/search with json={"q": query}
    # and headers={"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}.
    # Pull the top 3 results' "title" and "snippet" from response.json()["organic"]
    # and join them into one readable string, same pattern as search_flights /
    # search_hotels in the travel-agent notebooks.
    raise NotImplementedError("TODO 3: implement search_web")

### TODO 4 — wire all three tools into a LangGraph agent

This is the actual LangGraph lesson. Build:

1. A `State` (`TypedDict`) with one field, `messages`, accumulated via `add_messages` (same as `v3_toolnode.py`'s `State`)
2. An `agent` node that binds all three tools to the LLM and invokes it
3. A graph with **one** `ToolNode` wrapping **all three** tools — not three separate tool nodes
4. `tools_condition` to route between the agent and the tools, looping back after each tool call

Think back to the travel-agent assignment's B4 question before you write this: does having three tools instead of one change how many nodes you need?

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


# TODO 4a: build the list of all three tools, and bind them to the llm
# tools = [...]
# llm_with_tools = llm.bind_tools(tools)


def agent_node(state: State) -> dict:
    # TODO 4b: invoke llm_with_tools on state["messages"] and return
    # {"messages": [response]}
    raise NotImplementedError("TODO 4b: implement agent_node")


# TODO 4c: build the graph
#   - add_node("agent", agent_node)
#   - add_node("tools", ToolNode(tools))          <- ONE ToolNode, all 3 tools
#   - set the entry point to "agent"
#   - add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": END})
#   - add_edge("tools", "agent")                   <- loop back after any tool call
#   - compile it into `app`
app = None  # replace with your graph.compile() call

---
## Solution

Complete, runnable solution to TODOs 1-4. Re-running these cells will overwrite anything you defined above with the working versions.

### Solution 1 — `get_account_balance`

In [ ]:
@tool
def get_account_balance(account_number: str) -> str:
    """Look up a customer's name and balance by their bank account number (e.g. ACC1003)."""
    conn = sqlite3.connect("bank.db")
    c = conn.cursor()
    c.execute("""
    SELECT customer_name, balance
    FROM bank_accounts
    WHERE account_number = ?
    """, (account_number,))
    result = c.fetchone()
    conn.close()

    if result:
        customer_name, balance = result
        return f"Customer: {customer_name}, Balance: ${balance}"
    else:
        return "Account not found"


print(get_account_balance.invoke({"account_number": "ACC1003"}))

### Solution 2 — `search_bank_policies` (RAG)

In [ ]:
import math  # Import math module for square root calculations

# ============================================================
# FUNCTION 1: Calculate similarity between two vectors
# ============================================================
def cosine_similarity(a: list, b: list) -> float:
    """
    Calculates how similar two lists (vectors) are.
    Returns a number between 0 and 1:
    - 1.0 means they are identical (pointing in the same direction)
    - 0.0 means they are completely different (no similarity)
    
    Think of it like comparing two arrows:
    - If they point in the same direction → high similarity
    - If they point in opposite directions → low similarity
    """
    
    # STEP 1: Calculate the "dot product" (how much the vectors overlap)
    # zip(a, b) pairs up the first elements, second elements, etc.
    # Example: a = [1, 2], b = [3, 4]
    # zip gives: (1,3), (2,4)
    # x*y gives: 1*3=3, 2*4=8
    # sum gives: 3+8=11
    dot = sum(x * y for x, y in zip(a, b))
    
    # STEP 2: Calculate the "magnitude" (length) of vector 'a'
    # For a = [3, 4]: 3*3=9, 4*4=16, sum=25, sqrt(25)=5
    # This is like measuring how long the arrow is
    norm_a = math.sqrt(sum(x * x for x in a))
    
    # STEP 3: Calculate the "magnitude" (length) of vector 'b'
    # For b = [1, 2]: 1*1=1, 2*2=4, sum=5, sqrt(5)=2.236
    norm_b = math.sqrt(sum(y * y for y in b))
    
    # STEP 4: Calculate cosine similarity = dot product ÷ (length_a × length_b)
    # If both vectors have length > 0, calculate the ratio
    # If either is empty (length = 0), return 0.0 (no similarity)
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

# ============================================================
# FUNCTION 2: Search bank policies for relevant information
# ============================================================
@tool  # This decorator marks the function as a "tool" that can be used by an AI agent
def search_bank_policies(query: str) -> str:
    """
    Searches through bank policy documents to find the most relevant one
    based on the user's query.
    
    How it works:
    1. Convert the query into a vector (list of numbers) using an embedding model
    2. Compare the query vector to each policy's vector using cosine similarity
    3. Find the policy with the highest similarity score
    4. Return that policy's title and content
    """
    
    # STEP 1: Convert the user's question into a vector (list of numbers)
    # Embeddings turn text like "what happens if I overdraw?" into numbers like [0.12, -0.45, 0.78, ...]
    # This allows computers to compare text mathematically
    query_embedding = embeddings.embed_query(query)
    
    # STEP 2: Compare the query against every bank policy
    # For each policy in BANK_POLICIES, calculate its similarity to the query
    # Store the result as a list of tuples: (similarity_score, policy_data)
    scored = [
        (cosine_similarity(query_embedding, p["embedding"]), p)
        for p in BANK_POLICIES
    ]
    
    # STEP 3: Sort policies by similarity score (highest first)
    # scored.sort() would sort by first element, but we explicitly say "sort by pair[0]"
    # reverse=True means: largest score comes first
    scored.sort(key=lambda pair: pair[0], reverse=True)
    
    # STEP 4: Take the best matching policy (first item in sorted list)
    # best_score = the similarity score (e.g., 0.85)
    # best_policy = the policy dictionary containing title, content, embedding
    best_score, best_policy = scored[0]
    
    # STEP 5: Return the policy title and content as a formatted string
    return f"{best_policy['title']}: {best_policy['content']}"

# ============================================================
# MAIN: Test the function
# ============================================================
# This invokes the search tool with a question about overdraft
# The AI agent will search through bank policies and return the best match
print(search_bank_policies.invoke({"query": "what happens if I overdraw my account?"}))

| Code | What It Does | Example |
|------|--------------|---------|
| `math.sqrt()` | Calculates square root | `sqrt(25) = 5` |
| `zip(a, b)` | Pairs elements from two lists | `zip([1,2], [3,4])` → `[(1,3), (2,4)]` |
| `sum(x * y for x, y in zip(a, b))` | Dot product (multiply pairs and add) | `(1×3)+(2×4) = 11` |
| `norm_a` | Length of vector 'a' | `[3,4]` length = 5 |
| `cosine_similarity()` | Measures similarity between vectors | 0.92 = very similar |
| `@tool` | Makes function usable by AI agent | Agent can call it like a tool |
| `embed_query()` | Converts text to numbers | `"overdraft"` → `[0.12, -0.45]` |
| `scored.sort()` | Orders policies by relevance | Best match first |

 Real-World Analogy
Imagine you're a librarian with a giant file cabinet (BANK_POLICIES):

User asks: "What happens if I overdraw?"

You (the search_bank_policies function):

Write down the question on a sticky note (query_embedding)

Compare the sticky note to every file folder in the cabinet (cosine_similarity)

Find the folder that matches best (scored.sort())

Read that folder's title and content back to the user (return)

The cosine similarity is like measuring how well the sticky note's words match each folder's words. If they share many similar words, the score is high!

In [ ]:
User Query: "what happens if I overdraw my account?"
                    │
                    ▼
         ┌─────────────────────┐
         │  query_embedding    │  ← Convert to numbers: [0.12, -0.45, 0.78, ...]
         └─────────────────────┘
                    │
                    ▼
         ┌─────────────────────────────────────────────────────┐
         │  Compare with each BANK_POLICY                     │
         │                                                    │
         │  Policy 1: "Overdraft Policy"   Score: 0.92 ✅    │  ← BEST MATCH!
         │  Policy 2: "Account Closure"    Score: 0.31       │
         │  Policy 3: "Fraud Reporting"    Score: 0.15       │
         │  Policy 4: "Interest Rates"     Score: 0.08       │
         └─────────────────────────────────────────────────────┘
                    │
                    ▼
         ┌─────────────────────┐
         │  Sort scores        │
         │  1st: 0.92          │  ← Highest score
         │  2nd: 0.31          │
         │  3rd: 0.15          │
         │  4th: 0.08          │
         └─────────────────────┘
                    │
                    ▼
         ┌─────────────────────┐
         │  Return best match  │  → "Overdraft Policy: If you overdraw..."
         └─────────────────────┘

### Solution 3 — `search_web` (Serper)

In [ ]:
@tool
def search_web(query: str) -> str:
    """Search the live web for current information not contained in the bank's own records or policies -- e.g. current market interest rates, recent financial news, or general finance questions."""
    response = requests.post(
        "https://google.serper.dev/search",
        json={"q": query},
        headers={"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"},
        timeout=15,
    )
    data = response.json()

    results = [
        f"Title: {item.get('title')}\nSnippet: {item.get('snippet')}"
        for item in data.get("organic", [])[:3]
    ]
    return "\n\n".join(results) or "No results found."


print(search_web.invoke({"query": "current US federal funds interest rate"})[:300])

### Solution 4 — the LangGraph agent: one `ToolNode`, three tools

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


tools = [get_account_balance, search_bank_policies, search_web]
llm_with_tools = llm.bind_tools(tools)


def agent_node(state: State) -> dict:
    system = SystemMessage(content=(
        "You are a smart banking assistant with three tools:\n"
        "- get_account_balance: look up a specific account's balance by account number\n"
        "- search_bank_policies: answer questions about the bank's own policies "
        "(overdraft, closure, fraud, interest, dormant accounts)\n"
        "- search_web: answer general finance questions the bank's own data can't, "
        "like current market rates or financial news\n"
        "Use the single most appropriate tool for each question. Be concise and professional."
    ))
    response = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}


graph = StateGraph(State)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))   # <- ONE node, all 3 tools

graph.set_entry_point("agent")
graph.add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": END})
graph.add_edge("tools", "agent")

app = graph.compile()

print("✅ Graph compiled —", list(app.get_graph().nodes.keys()))
print("✅ Tools wrapped by the single ToolNode:", [t.name for t in tools])

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Visualization unavailable in this environment:", e)
    print(app.get_graph().draw_mermaid())

### Try it — three questions, three different tools, one agent

Watch which tool the agent picks for each question. You never told it explicitly which tool maps to which question type — that decision comes entirely from each tool's **docstring**, read by the LLM at call time.

In [ ]:
for question in [
    "What is the balance for account ACC1003?",
    "What happens if I overdraw my account?",
    "What is the current US federal funds interest rate?",
]:
    print(f"\n{'='*70}\nQ: {question}")
    result = app.invoke({"messages": [HumanMessage(content=question)]})
    print("A:", result["messages"][-1].content)

## What this assignment demonstrates

The graph has exactly **2 nodes** (`agent`, `tools`) regardless of whether it wraps 1 tool or 10 — adding `search_bank_policies` and `search_web` alongside `get_account_balance` never required a third or fourth node. This is the same B4 insight from the travel-agent ToolNode assignment, now demonstrated across three *structurally different* tool implementations at once:

| Tool | What it actually does internally | What `ToolNode` sees |
|---|---|---|
| `get_account_balance` | A SQLite query | A function with a name, args, and a docstring |
| `search_bank_policies` | An embedding call + cosine similarity loop (RAG) | A function with a name, args, and a docstring |
| `search_web` | An HTTP POST to Serper | A function with a name, args, and a docstring |

`ToolNode` never needed to know any of that — it only ever sees the *interface* (`@tool`-decorated, typed arguments, a docstring), never the implementation. That's the entire point of the abstraction: a database call, a RAG pipeline, and a live web search are completely interchangeable from the graph's point of view, which is exactly what makes LangGraph agents easy to extend — adding a fourth, completely different capability tomorrow means writing one more `@tool` function and adding it to one list, with zero changes to the graph itself.